In [1]:
import os
import subprocess
import pandas as pd
import numpy as np
import torch
from transformers import pipeline
from mmpose.apis import MMPoseInferencer
from tqdm.auto import tqdm 

/home/rehab/research/text2sign/.venv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/rehab/research/text2sign/.venv/lib/python3.8/site-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \


In [2]:
BASE_DIR = '' # เปลี่ยนเป็นพาทจริงของพี่
CSV_PATH = 'interpreter_dataset_info.csv'
OUTPUT_DIR = 'features'

In [3]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

# เช็ค GPU
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"✅ ใช้ Device: {DEVICE}")
if DEVICE == "cuda:0":
    print(f"🎮 การ์ดจอ: {torch.cuda.get_device_name(0)}")

✅ ใช้ Device: cuda:0
🎮 การ์ดจอ: NVIDIA GeForce RTX 4070 Ti SUPER


In [4]:
# [Cell 3 - Code]
def extract_audio(video_path, output_audio_path):
    """สกัดเสียงจากวิดีโอเป็น 16kHz Mono สำหรับ Whisper"""
    cmd = f"ffmpeg -i {video_path} -vn -acodec pcm_s16le -ar 16000 -ac 1 {output_audio_path} -loglevel quiet -y"
    subprocess.call(cmd, shell=True)

def crop_video(video_path, output_video_path, x1, y1, x2, y2):
    """ตัดกรอบวิดีโอเฉพาะตัวล่ามภาษามือ"""
    w, h = int(x2 - x1), int(y2 - y1)
    x, y = int(x1), int(y1)
    cmd = f"ffmpeg -i {video_path} -vf 'crop={w}:{h}:{x}:{y}' -c:v libx264 -crf 18 -preset fast -c:a copy {output_video_path} -loglevel quiet -y"
    subprocess.call(cmd, shell=True)

def cleanup_temp_files(*file_paths):
    """ลบไฟล์ชั่วคราวทิ้ง"""
    for path in file_paths:
        if os.path.exists(path):
            os.remove(path)

In [5]:
# [Cell 4 - Code]
print("⏳ กำลังโหลด Whisper (ASR) เข้า GPU...")
whisper_pipe = pipeline(
    "automatic-speech-recognition",
    model="biodatlab/whisper-th-large-v3-combined",
    device=DEVICE,
    chunk_length_s=30,
    batch_size=64, # อัด Batch Size ให้ 4070 Ti SUPER ดึงพลังให้สุด
    model_kwargs={"torch_dtype": torch.float16}
)
print("✅ โหลด Whisper สำเร็จ!")

print("\n⏳ กำลังโหลด MMPose (Wholebody) เข้า GPU...")
pose_inferencer = MMPoseInferencer(pose2d='wholebody', device=DEVICE)
print("✅ โหลด MMPose สำเร็จ!")

⏳ กำลังโหลด Whisper (ASR) เข้า GPU...


✅ โหลด Whisper สำเร็จ!

⏳ กำลังโหลด MMPose (Wholebody) เข้า GPU...
Loads checkpoint by http backend from path: https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/rtmpose-m_simcc-coco-wholebody_pt-aic-coco_270e-256x192-cd5e845c_20230123.pth
05/03 07:59:20 - mmengine - WARNING - Failed to search registry with scope "mmpose" in the "function" registry tree. As a workaround, the current "function" registry in "mmengine" is used to build instance. This may cause unexpected failure when running the built modules. Please check whether "mmpose" is a correct scope, or whether the registry is initialized.
Loads checkpoint by http backend from path: https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/rtmdet_m_8xb32-100e_coco-obj365-person-235e8209.pth
05/03 07:59:20 - mmengine - WARNING - Failed to search registry with scope "mmdet" in the "function" registry tree. As a workaround, the current "function" registry in "mmengine" is used to build instance. This may cause unexpected 

In [6]:
# [Cell 5 - Code สำหรับรันแบบ 4 ส่วน]

# --- 1. ตั้งค่าส่วนที่จะรัน (แก้ที่นี่ในแต่ละจอ) ---
PART_INDEX = 0  # เปลี่ยนเป็น 0, 1, 2, หรือ 3
DEVICE = "cuda:0" # จอ 0,1 ใช้ cuda:0 | จอ 2,3 ใช้ cuda:1

# --- 2. หั่นข้อมูล ---
full_df = pd.read_csv(CSV_PATH)
# แบ่งเป็น 4 ส่วนเท่าๆ กัน
parts = np.array_split(full_df, 4)
df = parts[PART_INDEX].reset_index(drop=True)

print(f"🚀 Part {PART_INDEX} | Device: {DEVICE} | ข้อมูล: {len(df)} แถว")

# --- 3. วนลูปประมวลผล ---
for index, row in tqdm(df.iterrows(), total=len(df), desc=f"Part {PART_INDEX}"):
    
    relative_path = str(row['Path']).strip()
    video_path = os.path.join(BASE_DIR, relative_path)
    
    if not os.path.exists(video_path) or pd.isna(row['x1']):
        continue

    name_only = os.path.splitext(os.path.basename(video_path))[0]
    output_npy_path = os.path.join(OUTPUT_DIR, f"{name_only}.npy")
    
    if os.path.exists(output_npy_path):
        continue

    # ⚡ ตั้งชื่อไฟล์ Temp ให้มี Part Index กันไฟล์ตีกันบน RAM
    temp_audio = f"/dev/shm/temp_p{PART_INDEX}_{name_only}.wav"
    temp_cropped_video = f"/dev/shm/temp_crop_p{PART_INDEX}_{name_only}.mp4"

    try:
        # --- A. ถอดเสียง ---
        extract_audio(video_path, temp_audio)
        asr_result = whisper_pipe(
            temp_audio, 
            return_timestamps=True,
            batch_size=16, # ปรับลดลงนิดนัยเพราะรัน 2 จอต่อการ์ด
            generate_kwargs={"language": "thai"}
        )
        
        # --- B. ตัดภาพ และ ดึงท่าทาง ---
        x1, y1, x2, y2 = float(row['x1']), float(row['y1']), float(row['x2']), float(row['y2'])
        crop_video(video_path, temp_cropped_video, x1, y1, x2, y2)

        # ⚡ Batch Size 128 (รวม 2 จอเป็น 256 ต่อการ์ด 1 ใบ)
        result_generator = pose_inferencer(
            temp_cropped_video, 
            return_vis=False, 
            batch_size=128,
            device=DEVICE # บังคับให้ใช้การ์ดจอตามที่ตั้งไว้
        )
        
        video_keypoints = [res['predictions'][0][0]['keypoints'] if len(res['predictions']) > 0 
                           else np.zeros((133, 2)) for res in result_generator]
                
        # --- C. แพ็กรวม ---
        np.save(output_npy_path, {
            'keypoints': np.array(video_keypoints),
            'asr': asr_result["text"].strip(),
            'asr_chunks': asr_result["chunks"],
            'crop_box': [x1, y1, x2, y2],
            'original_path': relative_path
        }, allow_pickle=True)

    except Exception as e:
        print(f"\n❌ Error Part {PART_INDEX} @ {name_only}: {e}")
    finally:
        cleanup_temp_files(temp_audio, temp_cropped_video)

🚀 Part 0 | Device: cuda:0 | ข้อมูล: 75 แถว


Part 0:   0%|          | 0/75 [00:00<?, ?it/s]/home/rehab/research/text2sign/.venv/lib/python3.8/site-packages/transformers/models/whisper/generation_whisper.py:509: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
You have passed language=thai, but also have set `forced_decoder_ids` to [[1, None], [2, 50360]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of language=thai.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.
/home/rehab/research/text2sign/.venv/lib/python3.8/site-packages/mmdet/models/layers/se_layer.py:158: FutureWarning

KeyboardInterrupt: 